### Step 3 - Novelty Scoring (Unsupervised, Temporal) - Feature Construction Stage

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

- STEP 3B — Citation Features
- Extract:
- cited_by_count
- reference_count

In [2]:
meta = pd.read_csv("../../outputs/intermediate/openalex_metadata_full.csv")

citation_df = meta[[
    "global_paper_id",
    "cited_by_count",
    "referenced_works",
    "year"
]].copy()


def count_refs(x):
    if pd.isna(x):
        return 0
    try:
        return len(eval(x))
    except:
        return 0


citation_df["reference_count"] = citation_df["referenced_works"].apply(count_refs)

citation_df = citation_df.rename(columns={
    "global_paper_id": "paper_id"
})

citation_df = citation_df[[
    "paper_id",
    "year",
    "cited_by_count",
    "reference_count"
]]

citation_df.to_csv("../../outputs/other/citation_features.csv", index=False)

print("Citation features saved.")

Citation features saved.


- STEP 3C — Feature Matrix Construction
- Combine:
- Semantic novelty (kNN)
- Structural novelty
- Citation features

In [3]:
semantic_df = pd.read_csv("../../outputs/final/semantic_novelty_knn_scores.csv")
struct_df = pd.read_csv("../../outputs/final/structural_novelty_scores.csv")
citation_df = pd.read_csv("../../outputs/other/citation_features.csv")

# Merge
features = semantic_df.merge(struct_df,
                             on="paper_id",
                             how="left")

features = features.merge(citation_df,
                          on=["paper_id", "year"],
                          how="left")

features = features.fillna(0)

print("Feature matrix shape:", features.shape)

features.to_csv("../../outputs/other/novelty_feature_matrix.csv", index=False)

Feature matrix shape: (4242, 7)


- STEP 3D — Composite Novelty Score (Continuous)
- Instead of classification, we define:
-   composite_novelty =
-       w1 * structural +
-       w2 * semantic +
-       w3 * citation_signal
- Citation signal normalized.

In [4]:
features = pd.read_csv("../../outputs/other/novelty_feature_matrix.csv")

# Normalize citation signal (log transform for stability)
features["citation_signal"] = np.log1p(features["cited_by_count"])


# Min-max normalize components
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)


features["struct_norm"] = minmax(features["structural_novelty"])
features["semantic_norm"] = minmax(features["semantic_knn"])
features["citation_norm"] = minmax(features["citation_signal"])

# Weighted combination (can tune later)
features["composite_novelty"] = (
        0.20 * features["struct_norm"] +
        0.70 * features["semantic_norm"] +
        0.10 * features["citation_norm"]
)

features.to_csv("../../outputs/final/novelty_feature_matrix_with_score.csv",
                index=False)

print("Composite novelty score computed.")

Composite novelty score computed.


In [5]:
pd.read_csv('../../outputs/final/novelty_feature_matrix_with_score.csv').columns

Index(['paper_id', 'year', 'semantic_knn', 'structural_novelty',
       'triple_count', 'cited_by_count', 'reference_count', 'citation_signal',
       'struct_norm', 'semantic_norm', 'citation_norm', 'composite_novelty'],
      dtype='object')

In [8]:
fm = pd.read_csv('../../outputs/final/novelty_feature_matrix_with_score.csv')
fm.head()

,paper_id,year,semantic_knn,structural_novelty,triple_count,cited_by_count,reference_count,citation_signal,struct_norm,semantic_norm,citation_norm,composite_novelty
0,SKG_MT_1151,2010,1.0,1.0,70.0,3,17,1.386294,1.0,1.0,0.149105,0.914911
1,SKG_SUM_67,2010,1.0,1.0,68.0,37,16,3.637586,1.0,1.0,0.391246,0.939125
2,SKG_MT_287,2010,1.0,1.0,85.0,0,14,0.000000,1.0,1.0,0.000000,0.900000
3,SKG_SUM_83,2010,1.0,1.0,35.0,56,18,4.043051,1.0,1.0,0.434857,0.943486
4,SKG_SUM_42,2010,1.0,1.0,89.0,18,31,2.944439,1.0,1.0,0.316694,0.931669


In [9]:
features[['semantic_knn', 'structural_novelty']].corr()

,semantic_knn,structural_novelty
semantic_knn,1.000000,0.331567
structural_novelty,0.331567,1.000000


In [10]:
df_clean = fm[fm['year']>2010]
df_clean = df_clean[df_clean['triple_count']>0]
df_clean.head()

,paper_id,year,semantic_knn,structural_novelty,triple_count,cited_by_count,reference_count,citation_signal,struct_norm,semantic_norm,citation_norm,composite_novelty
59,SKG_MT_620,2011,0.922090,1.000000,2.0,6,17,1.945910,1.000000,0.818615,0.209295,0.793960
60,SKG_MT_964,2011,0.930110,0.853659,41.0,4,13,1.609438,0.853659,0.837286,0.173106,0.774143
61,SKG_MT_966,2011,0.916807,0.916667,12.0,17,10,2.890372,0.916667,0.806314,0.310878,0.778841
62,SKG_MT_1227,2011,0.896374,0.888889,9.0,56,15,4.043051,0.888889,0.758743,0.434857,0.752384
63,SKG_MT_1220,2011,0.933532,0.881818,110.0,2,23,1.098612,0.881818,0.845252,0.118163,0.779856


In [11]:
df_clean.groupby('year').size()

year
2011     64
2012     73
2013    122
2014     89
2015    116
2016    129
2017    221
2018    290
2019    436
2020    504
2021    403
2022    255
2023    294
2024     28
2025      3
dtype: int64

In [12]:
from scipy import stats
r, p = stats.pearsonr(df_clean['semantic_knn'], df_clean['structural_novelty'])

In [13]:
print(f"Clean correlation: r={r:.4f}, p={p:.4f}, n={len(df_clean)}")

Clean correlation: r=-0.3549, p=0.0000, n=3027


In [14]:
pn = pd.read_csv('../../outputs/final/paper_nodes.csv')
pn.head()

,node_id,node_type,title,domain,split,year,openalex_id,cited_by_count,score,method
0,NOVEL_DIA_0,Paper,MRF-Chat Improving Dialogue with Markov Random...,DIA,NOVEL,2021.0,https://openalex.org/W3214342458,0.0,0.606481,tfidf
1,NOVEL_DIA_1,Paper,Towards Making the Most of Dialogue Characteri...,DIA,NOVEL,2021.0,https://openalex.org/W3196896228,12.0,0.663980,tfidf
2,NOVEL_DIA_2,Paper,Domain-Adaptive Pretraining Methods for Dialog...,DIA,NOVEL,2021.0,https://openalex.org/W3173606101,19.0,0.668112,tfidf
3,NOVEL_DIA_3,Paper,Adaptive Bridge between Training and Inference...,DIA,NOVEL,2021.0,https://openalex.org/W3214623240,5.0,0.636847,tfidf
4,NOVEL_DIA_4,Paper,Controlling Dialogue Generation with Semantic ...,DIA,NOVEL,2021.0,https://openalex.org/W3074476581,6.0,0.811583,tfidf


In [15]:
df = fm.merge(pn[['node_id','split','domain']], left_on='paper_id', right_on='node_id', how='left')
# df = df[df['split'] == 'NOVEL']
# df.set_index('split')
df.groupby('split')[['struct_norm', 'semantic_norm', 'composite_novelty']].mean()

,struct_norm,semantic_norm,composite_novelty
split,,,
BLOG,0.292680,0.763072,0.657300
NOVEL,0.467926,0.837444,0.725516
SKG,0.641160,0.838900,0.759364
